In [1]:
%load_ext autoreload
%autoreload 2

In [24]:
from enum import Enum
from pathlib import Path

import polars as pl

In [3]:
data_dir = Path("./extdata")

In [4]:
def make_df(name: str) -> pl.DataFrame:
    file_path = data_dir / f"{name}.dbn.parquet"
    print(f"Loading {file_path}...")
    return pl.read_parquet(file_path)

In [5]:
df = make_df("xnas-itch-20260126.mbp-10")

Loading extdata/xnas-itch-20260126.mbp-10.dbn.parquet...


In [6]:
spy = df.filter(pl.col("symbol") == "SPY")

In [ ]:
show_df = spy.select(["ts_event", "bid_px_00", "bid_sz_00", "ask_px_00", "ask_sz_00"])

In [8]:
print(spy.columns)

['ts_event', 'rtype', 'publisher_id', 'instrument_id', 'action', 'side', 'depth', 'price', 'size', 'flags', 'ts_in_delta', 'sequence', 'bid_px_00', 'ask_px_00', 'bid_sz_00', 'ask_sz_00', 'bid_ct_00', 'ask_ct_00', 'bid_px_01', 'ask_px_01', 'bid_sz_01', 'ask_sz_01', 'bid_ct_01', 'ask_ct_01', 'bid_px_02', 'ask_px_02', 'bid_sz_02', 'ask_sz_02', 'bid_ct_02', 'ask_ct_02', 'bid_px_03', 'ask_px_03', 'bid_sz_03', 'ask_sz_03', 'bid_ct_03', 'ask_ct_03', 'bid_px_04', 'ask_px_04', 'bid_sz_04', 'ask_sz_04', 'bid_ct_04', 'ask_ct_04', 'bid_px_05', 'ask_px_05', 'bid_sz_05', 'ask_sz_05', 'bid_ct_05', 'ask_ct_05', 'bid_px_06', 'ask_px_06', 'bid_sz_06', 'ask_sz_06', 'bid_ct_06', 'ask_ct_06', 'bid_px_07', 'ask_px_07', 'bid_sz_07', 'ask_sz_07', 'bid_ct_07', 'ask_ct_07', 'bid_px_08', 'ask_px_08', 'bid_sz_08', 'ask_sz_08', 'bid_ct_08', 'ask_ct_08', 'bid_px_09', 'ask_px_09', 'bid_sz_09', 'ask_sz_09', 'bid_ct_09', 'ask_ct_09', 'symbol']


In [ ]:
show_df = show_df.with_columns(spread=pl.col("ask_px_00") - pl.col("bid_px_00"))
print(show_df.head(20))

shape: (20, 6)
┌─────────────────────────────────┬───────────┬───────────┬───────────┬───────────┬────────┐
│ ts_event                        ┆ bid_px_00 ┆ bid_sz_00 ┆ ask_px_00 ┆ ask_sz_00 ┆ spread │
│ ---                             ┆ ---       ┆ ---       ┆ ---       ┆ ---       ┆ ---    │
│ datetime[ns, UTC]               ┆ f64       ┆ u32       ┆ f64       ┆ u32       ┆ f64    │
╞═════════════════════════════════╪═══════════╪═══════════╪═══════════╪═══════════╪════════╡
│ 2026-01-26 09:00:00.000452732 … ┆ 689.47    ┆ 100       ┆ null      ┆ 0         ┆ null   │
│ 2026-01-26 09:00:00.000596693 … ┆ 689.47    ┆ 100       ┆ 689.56    ┆ 100       ┆ 0.09   │
│ 2026-01-26 09:00:00.030085797 … ┆ 689.47    ┆ 100       ┆ 689.56    ┆ 100       ┆ 0.09   │
│ 2026-01-26 09:00:00.030193828 … ┆ 689.47    ┆ 100       ┆ 689.55    ┆ 120       ┆ 0.08   │
│ 2026-01-26 09:00:00.031390574 … ┆ 689.47    ┆ 100       ┆ 689.55    ┆ 120       ┆ 0.08   │
│ …                               ┆ …         ┆ …      

In [20]:
show_df.describe()

statistic,ts_event,bid_px_00,bid_sz_00,ask_px_00,ask_sz_00,spread
str,str,f64,f64,f64,f64,f64
"""count""","""4929333""",4.929333e6,4.929333e6,4.929332e6,4.929333e6,4.929332e6
"""null_count""","""0""",0.0,0.0,1.0,0.0,1.0
"""mean""","""2026-01-26 16:46:08.577627+00:…",692.653793,480.157861,692.668453,490.796283,0.014659
"""std""",null,1.165542,364.614592,1.157873,382.130888,0.016276
"""min""","""2026-01-26 09:00:00.000452+00:…",685.0,1.0,686.96,0.0,0.01
"""25%""","""2026-01-26 15:03:27.835762+00:…",692.33,260.0,692.35,240.0,0.01
"""50%""","""2026-01-26 16:12:08.211153+00:…",692.92,400.0,692.94,400.0,0.01
"""75%""","""2026-01-26 18:36:59.290198+00:…",693.46,580.0,693.47,600.0,0.01
"""max""","""2026-01-26 23:59:59.996229+00:…",694.13,11100.0,694.15,8190.0,4.54


In [ ]:
print(f"Avg. spread: {show_df.select(pl.col('spread').mean())[0, 0]}")

Avg. spread: 0.014658921330517376


In [ ]:
class OrderSide(Enum):
    BUY = "buy"
    SELL = "sell"


def simulate_fill(
    order_side: OrderSide,
    order_size: int,
    bid_px: float,
    ask_px: float,
    bid_sz: int,
    ask_sz: int,
) -> tuple[float, int]:
    if order_side == OrderSide.BUY:
        if order_size <= ask_sz:
            return ask_px, order_size
        else:
            return ask_px, ask_sz
    elif order_side == OrderSide.SELL:
        if order_size <= bid_sz:
            return bid_px, order_size
        else:
            return bid_px, bid_sz

Number of fills with max spread 0.05: 4773142
